In [1]:
from datasets import load_dataset

ds = load_dataset("phunc20/nj_biergarten_captcha")

c:\Users\krist\Documents\UNI\8sem\TAI\Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [27]:
# Print the key of the first example in the training set
print(ds["train"][0]["__key__"])
#Show the image
ds["train"][0]["jpg"].show()
# Extract the label by extracting characters after '_'
print(*ds["train"][0]["__key__"].split("_")[1:])

001/2024-10-08T18:21:10.578994_y2jhuv
y2jhuv


In [3]:
import numpy as np
import torch
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split

* Functions

In [15]:
def extract_labels(dataset):
    # Extract the labels from the dataset
    return np.array([key.split("_")[1:] for key in dataset["__key__"]]).flatten()

class CustomCaptchaDataset(Dataset):
    def __init__(self, dataset, transform=None):
        self.transform = transform
        self.dataset = dataset
        
        # dont read .mat files present in the folder\n",
        self.image_files = [img for img in dataset["jpg"]]
        # Convert image from jpg files to numpy arrays
        # Remove flatten
        self.image_files = np.array([np.array(image) for image in self.image_files])
        self.labels = extract_labels(dataset)

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        label = self.labels[idx]
        image = self.image_files[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

In [32]:
data_points = 1000
batch_size = 10

train_size = int(data_points * 0.75)
test_size = int(data_points - train_size)
print(f"Train size: {train_size}, Test size: {test_size}")

dataset = CustomCaptchaDataset(ds["train"][:data_points], transform=transforms.ToTensor())

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

# Create DataLoaders for batch processing\n",
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(dataset[0][0].shape)
print(dataset[0][1])


Train size: 750, Test size: 250
torch.Size([3, 50, 140])
y2jhuv
